### Как реализовать OOT-тест в коде

In [ ]:
# --- 1. Финальное разбиение ---
# Данные предоставлены в df
# Выделяем 30% для финального OOT теста.
test_size = 30 
train_val_size = len(df) - test_size

# Данные для обучения и валидации (Train_Val)
X_train_val = X[:train_val_size]
y_train_val = y[:train_val_size]
df_train_val = df.iloc[:train_val_size]

# Данные для финального OOT-теста
X_test_oot = X[train_val_size:]
y_test_oot = y[train_val_size:]
df_test_oot = df.iloc[train_val_size:]

print(f"Размер обучения и валидации: {len(X_train_val)} точек (до {df_train_val.index[-1].date()})")
print(f"Размер OOT Test: {len(X_test_oot)} точек (с {df_test_oot.index[0].date()} по {df_test_oot.index[-1].date()})")

# --- 2. Валидация ---
# Здесь мы используем TimeSeriesSplit, чтобы настроить модель
tscv = TimeSeriesSplit(n_splits=3) 
tscv_mse_scores = []

for train_index, val_index in tscv.split(X_train_val):
    X_train_cv, X_val_cv = X_train_val[train_index], X_train_val[val_index]
    y_train_cv, y_val_cv = y_train_val[train_index], y_train_val[val_index]

    model_cv = RandomForestRegressor(n_estimators=50, random_state=42, max_depth=100)
    model_cv.fit(X_train_cv, y_train_cv)
    
    y_val_pred = model_cv.predict(X_val_cv)
    tscv_mse = mean_squared_error(y_val_cv, y_val_pred)
    tscv_mse_scores.append(tscv_mse)

print(f"\nСредняя MSE TSCV (настройка): {np.mean(tscv_mse_scores):.4f}")


# --- 3. Финальное обучение и OOT-тест ---
# Обучаем финальную модель на всех данных до OOT
final_model = RandomForestRegressor(n_estimators=50, random_state=42)
final_model.fit(X_train_val, y_train_val)

# Прогноз на финальном OOT-наборе
y_oot_pred = final_model.predict(X_test_oot)
final_oot_mse = mean_squared_error(y_test_oot, y_oot_pred)

print(f"\nФИНАЛЬНЫЙ OOT-TEST MSE: {final_oot_mse:.4f}")


# --- 4. Визуализация финального разбиения и прогноза ---
plt.figure(figsize=(14, 6))

# Train/Validation
plt.plot(df_train_val.index, df_train_val['Value'], label='Обучение + Валидация (In-Time)', color='#1f77b4', linewidth=2)

# OOT Test (Фактические значения)
plt.plot(df_test_oot.index, df_test_oot['Value'], label='Финальный OOT Тест (Actual)', color='#ff7f0e', linewidth=2)

# OOT Прогноз
plt.plot(df_test_oot.index, y_oot_pred, label='OOT Прогноз (Prediction)', color='green', linestyle='--', linewidth=2.5)

# Линия разбиения
plt.axvline(x=df_test_oot.index[0], color='red', linestyle='-', linewidth=1, label='Разбиение Train/Test')

plt.title('Финальное разбиение и прогноз OOT (Out-of-Time) Test')
plt.xlabel('Дата')
plt.ylabel('Значение')
plt.legend(loc='upper left')
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()